# Indonesia International Tourist Arrivals Analysis
**Data Source:** Badan Pusat Statistik (BPS) — bps.go.id  
**Period:** 2023 – 2025  
**Tools:** Python, pandas, matplotlib


## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Load & Preview Raw Data
The file has a tricky 3-row header (title / year / month) because of merged cells in Excel.  
We load it **without** a header first so we can see the raw structure.

In [ ]:
FILE = 'Query_Builder_Result_-_Rabu__08_April_2026_pukul_14_59_27_WIB.xlsx'

# Load without header
raw = pd.read_excel(FILE, header=None)

print('Shape:', raw.shape)
raw.iloc[:4, :6]  # Preview first 4 rows, first 6 columns

## 3. Data Cleaning
### What problems did we find?
| Problem | Fix |
|---------|-----|
| 3-row merged header | Rebuild column names manually |
| Subtotal rows mixed in | Remove section headers & grand totals |
| Dash `-` used instead of 0 | Replace with 0 |
| Missing values (NaN) | Fill with 0 |

In [ ]:
# ── Step 1: Rebuild column names from the 3-row header ──
year_row  = raw.iloc[1].tolist()
month_row = raw.iloc[2].tolist()

# Forward-fill year labels (merged cells leave NaN gaps)
current_year = None
years_filled = []
for v in year_row:
    if str(v).strip() in ['2023', '2024', '2025']:
        current_year = int(float(str(v)))
    years_filled.append(current_year)

# Combine year + month into one column name
col_names = []
for i, (yr, mo) in enumerate(zip(years_filled, month_row)):
    if i == 0:
        col_names.append('port')
    elif pd.notna(mo):
        col_names.append(f"{yr}_{str(mo).strip()}")
    else:
        col_names.append(f"col_{i}")

print('Sample column names:', col_names[:5])

In [ ]:
# ── Step 2: Extract only data rows (skip header rows & footer) ──
df = raw.iloc[3:38].copy()
df.columns = col_names

# ── Step 3: Remove subtotal & grand total rows ──
REMOVE = ['A. Pintu Udara', 'B. Pintu Laut', 'C. Pintu Darat',
          'Jumlah (A+B+C)', 'Lainnya', 'Total']
df = df[~df['port'].isin(REMOVE)]
df = df[df['port'].notna()].copy()

print(f'Entry points after cleaning: {len(df)}')
df[['port']].reset_index(drop=True)

In [ ]:
# ── Step 4: Fix dirty values (dash → 0, NaN → 0, cast to int) ──
value_cols = [c for c in df.columns if c != 'port']

for col in value_cols:
    df[col] = (
        df[col]
        .replace('-', 0)
        .fillna(0)
        .apply(lambda x: int(float(str(x).replace(',', ''))))
    )

print('All values cleaned and converted to integer!')
df.head(3)

In [ ]:
# ── Step 5: Add entry type label (Air / Sea / Land) ──
AIR  = ['Ngurah Rai','Soekarno-Hatta','Juanda','Kualanamu','Husein Sastranegara',
         'Adi Sucipto','Bandara Int. Lombok','Sam Ratulangi','Minangkabau',
         'Sultan Syarif Kasim II','Sultan Iskandar Muda','Ahmad Yani','Supadio',
         'Hasanuddin','Sultan Badaruddin II','Pintu Udara Lainnya']
SEA  = ['Batam','Tanjung Uban','Tanjung Pinang','Tanjung Balai Karimun',
         'Tanjung Benoa','Tanjung Mas','Pintu Laut Lainnya']

def entry_type(name):
    if name in AIR:  return 'Air'
    if name in SEA:  return 'Sea'
    return 'Land'

df['entry_type'] = df['port'].apply(entry_type)
df['entry_type'].value_counts()

## 4. Reshape to Tidy Format
The raw data is **wide** (one column per month). We convert it to **long** format:  
one row per `(port, year, month)` — this is the standard format for analysis and Power BI.

In [ ]:
MONTHS = {
    'Januari':'Jan','Februari':'Feb','Maret':'Mar','April':'Apr',
    'Mei':'May','Juni':'Jun','Juli':'Jul','Agustus':'Aug',
    'September':'Sep','Oktober':'Oct','November':'Nov','Desember':'Dec'
}
MONTH_NUM = {v: i+1 for i, v in enumerate(MONTHS.values())}

records = []
for _, row in df.iterrows():
    for year in [2023, 2024, 2025]:
        for mo_id, mo_en in MONTHS.items():
            col = f"{year}_{mo_id}"
            if col in df.columns:
                records.append({
                    'port'      : row['port'],
                    'entry_type': row['entry_type'],
                    'year'      : year,
                    'month'     : MONTH_NUM[mo_en],
                    'month_name': mo_en,
                    'arrivals'  : row[col]
                })

tidy = pd.DataFrame(records)
print('Tidy dataframe shape:', tidy.shape)
tidy.head(6)

## 5. Export Clean Data

In [ ]:
# Long format — best for Power BI
tidy.to_csv('indonesia_tourism_clean.csv', index=False)

print('Exported: indonesia_tourism_clean.csv')
print(f'Rows: {len(tidy):,}  |  Columns: {list(tidy.columns)}')

## 6. Exploratory Data Analysis (EDA)

### 6.1 Overall Annual Totals

In [ ]:
annual = tidy.groupby('year')['arrivals'].sum()

for year in [2023, 2024, 2025]:
    print(f"{year}: {annual[year]:>12,}")

print()
print(f"Growth 2023→2024: +{(annual[2024]-annual[2023])/annual[2023]*100:.1f}%")
print(f"Growth 2024→2025: +{(annual[2025]-annual[2024])/annual[2024]*100:.1f}%")

### 6.2 Top 5 Ports by 2025 Arrivals

In [ ]:
port_year = tidy.groupby(['port','entry_type','year'])['arrivals'].sum().reset_index()
pivot = port_year.pivot_table(index=['port','entry_type'], columns='year',
                               values='arrivals', fill_value=0).reset_index()
pivot.columns.name = None

# Calculate growth
pivot['growth_24_25'] = ((pivot[2025] - pivot[2024]) / pivot[2024] * 100).round(1)

top5 = pivot.nlargest(5, 2025)[['port','entry_type',2023,2024,2025,'growth_24_25']]
top5.reset_index(drop=True)

### 6.3 Bali Concentration — How dependent is Indonesia on one port?

In [ ]:
bali = pivot[pivot['port'] == 'Ngurah Rai']
total_by_year = tidy.groupby('year')['arrivals'].sum()

print('Ngurah Rai (Bali) share of total arrivals:')
for year in [2023, 2024, 2025]:
    bali_v = int(bali[year].values[0])
    total_v = total_by_year[year]
    print(f"  {year}: {bali_v:,} / {total_v:,} = {bali_v/total_v*100:.1f}%")

### 6.4 Seasonality — Which months are busiest?

In [ ]:
monthly = tidy.groupby(['year','month','month_name'])['arrivals'].sum().reset_index()

# Seasonality index: 100 = average month
print('Seasonality index by month (100 = monthly average):')
print()
for year in [2023, 2024, 2025]:
    sub = monthly[monthly['year'] == year].sort_values('month')
    mean = sub['arrivals'].mean()
    index = (sub['arrivals'] / mean * 100).round(0).astype(int).tolist()
    labels = sub['month_name'].tolist()
    print(f"{year}:")
    for label, idx in zip(labels, index):
        bar = '█' * (idx // 10)
        print(f"  {label:>3}: {idx:>3}  {bar}")
    print()

## 7. Visualizations
We build **4 charts** — each one answers a specific question.

### Chart 1 — Monthly Trend: Is tourism growing?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = {2023: '#378ADD', 2024: '#1D9E75', 2025: '#D4537E'}
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for year in [2023, 2024, 2025]:
    sub = monthly[monthly['year'] == year].sort_values('month')
    ax.plot(sub['month'], sub['arrivals'] / 1e6,
            color=colors[year], linewidth=2.5,
            marker='o', markersize=5, label=str(year))

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))
ax.set_title('Monthly International Tourist Arrivals (2023–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Arrivals (millions)')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('chart1_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 2 — Entry Type: Which mode of travel is growing?

In [ ]:
type_year = tidy.groupby(['entry_type','year'])['arrivals'].sum().reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
type_colors = {'Air': '#378ADD', 'Sea': '#1D9E75', 'Land': '#D4537E'}
import numpy as np
bottoms = np.zeros(3)
years = [2023, 2024, 2025]

for etype in ['Air', 'Sea', 'Land']:
    vals = [type_year[(type_year['entry_type']==etype) &
                      (type_year['year']==y)]['arrivals'].sum() / 1e6
            for y in years]
    bars = ax.bar(years, vals, bottom=bottoms,
                  color=type_colors[etype], label=etype, width=0.5)
    for bar, val, bot in zip(bars, vals, bottoms):
        if val > 0.3:
            ax.text(bar.get_x() + bar.get_width()/2, bot + val/2,
                    f'{val:.1f}M', ha='center', va='center',
                    color='white', fontsize=9, fontweight='bold')
    bottoms += np.array(vals)

ax.set_xticks(years)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.set_title('Arrivals by Entry Type (2023–2025)', fontsize=14, fontweight='bold')
ax.set_ylabel('Arrivals (millions)')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('chart2_entry_type.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 3 — Top 10 Ports: Where do tourists enter?

In [ ]:
top10 = pivot.nlargest(10, 2025).sort_values(2025)
bar_colors = [type_colors[t] for t in top10['entry_type']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['port'], top10[2025] / 1e6, color=bar_colors, height=0.6)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.03, bar.get_y() + bar.get_height()/2,
            f'{w:.2f}M', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=type_colors[t], label=t) for t in ['Air','Sea','Land']]
ax.legend(handles=legend, frameon=False)
ax.set_xlabel('2025 Annual Arrivals (millions)')
ax.set_title('Top 10 Entry Points — 2025', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('chart3_top10_ports.png', dpi=150, bbox_inches='tight')
plt.show()

### Chart 4 — Bali Dependency: Is Indonesia over-reliant on one destination?

In [ ]:
bali_vals  = [int(bali[y].values[0]) for y in [2023,2024,2025]]
total_vals = [total_by_year[y] for y in [2023,2024,2025]]
other_vals = [t - b for t, b in zip(total_vals, bali_vals)]
bali_pct   = [b/t*100 for b, t in zip(bali_vals, total_vals)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: side-by-side bar
x = [2023, 2024, 2025]
ax1.bar([v - 0.2 for v in x], [v/1e6 for v in bali_vals],
        width=0.35, color='#378ADD', label='Ngurah Rai (Bali)')
ax1.bar([v + 0.2 for v in x], [v/1e6 for v in other_vals],
        width=0.35, color='#888780', label='All other ports')
ax1.set_xticks(x)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}M'))
ax1.set_title('Bali vs Rest of Indonesia', fontsize=12, fontweight='bold')
ax1.legend(frameon=False, fontsize=9)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: Bali share %
bars = ax2.bar(x, bali_pct, color='#378ADD', width=0.4)
for bar, pct in zip(bars, bali_pct):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 60)
ax2.set_xticks(x)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax2.set_title("Bali's share of total arrivals", fontsize=12, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Concentration Risk: Bali Dependency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart4_bali_dependency.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Insights

### 1. Tourism is growing but slowing down
- 2023 → 2024: **+18.9%** (strong rebound)
- 2024 → 2025: **+10.8%** (still growing, but pace is slowing)
- This is typical post-pandemic recovery — early years see big jumps, then it normalizes

### 2. Bali handles ~45% of ALL international arrivals — every year
- The share is almost unchanged: 44.9% (2023), 45.4% (2024), 44.9% (2025)
- This is a **concentration risk** — if Bali has any disruption, Indonesia loses nearly half its tourism

### 3. July and August are always the peak months
- Consistently 15–17% above the monthly average
- February and March are always the slowest
- Stable seasonality = good for planning hotel capacity, flights, staffing

### 4. Land borders are the fastest-growing entry type
- Land grew **+53.7%** over 2 years (vs Air +35.7%, Sea +12%)
- **Atambua** (East Nusa Tenggara border with East Timor) grew **+60% in 2025 alone**

### 5. Batam is growing faster than Jakarta's airport
- Batam: **+21.1%** in 2025 | Soekarno-Hatta: **+9.4%**
- Singapore proximity is driving short-trip cross-border tourism

---
*Analysis by [Bariq Jauhar Rizqullah] · Data: BPS Statistics Indonesia (bps.go.id)*